In [ ]:
options(encoding = "UTF-8")
Sys.setlocale("LC_ALL", "pl_PL.UTF-8")

# packages loader
use_package <- function(pkg) {
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg)
    library(pkg, character.only = TRUE)
  }
}

use_package("MASS")
use_package("readxl") # not working
use_package("gdata") # not working
use_package("FKSUM")
use_package('AER')
use_package('car')
use_package('lmtest')
use_package('splines')

Zadanie 1

In [ ]:
# generacja danyc
x = 1:20
y = x + rnorm(20)
X = cbind(1, x) # to są współczynniki, które będziemy dopasowaywać, odpowiednie potęgi

# dopasowanie modelu
model_lm = lm(y~x)
beta = solve(t(X) %*% X) %*% t(X) %*% y

model_poly = lm(y ~ poly(x, 3))
# to samo dla bety można 

In [ ]:
print(coef(model_lm))
print(beta)

Zadanie 2

In [ ]:
# loading data
data('CPS1988')
summary(CPS1988)

# 1. model training 
print('----Zadanie 1---- ')
cps_lm <- lm(log(wage)~experience + I(experience^2)+education + ethnicity, data=CPS1988)
summary(cps_lm)
# nie rozumiem pytania które jest zadane]

# 2.  test za pomocą anowy
print(' ----zadanie 2---- ')
cps_noeth <- lm(log(wage)~experience + I(experience^2)+education,
data = CPS1988)
print(anova(cps_noeth,cps_lm)) # daje informacje o tym który model jest bardziej istotny (rozważamy istotność modeli)
print(anova(cps_lm)) # daje informacje o tym które zmienne są mnie jważne  (czyli rozważamy istotność parametrów )

# 3. poza informacja o etniczności 
print(' ----zadanie 3---- ')
cps_noeth2 <- update(cps_lm, formula = .~.-ethnicity)
print(anova(cps_noeth2, cps_noeth))

# 4. Testy walda
print(' ----zadanie 4---- ')
print(waldtest(cps_lm,.~.-ethnicity))

Zadanie 3

In [ ]:
# tutaj analiza danych konkretnych 

Zadanie 4

Parametr K do AIC - możemy ją dostosowywać, bo otrzymujemy, wtedy, że 

In [ ]:
#AIC
# basic model
cps_plm <- lm(log(wage)~bs(experience,df=5)+education+ethnicity,data=CPS1988)

# 2. we are choosing best model using spline with AIC score
print('----Zadanie 2----')
cps_bs<-lapply(3:10,function(i) lm(log(wage)~bs(experience,df=i)
+education+ethnicity,data=CPS1988))
structure(sapply(cps_bs,AIC,k=log(nrow(CPS1988))),.Names=3:10)
# no to nam mówi który można wybrać, trzeba dać argmin
# 3. 
print('----Zadanie 3')
cps<-data.frame(experience=-2:60, education=with(CPS1988,
mean(education[ethnicity=="cauc"])),ethnicity="cauc")
cps$yhat1<-predict(cps_lm,newdata=cps)
cps$yhat2<-predict(cps_plm,newdata=cps)
plot(log(wage)~jitter(experience,factor=3),pch=19,
col=rgb(0.5,0.5,0.5,alpha=0.02),data=CPS1988)
lines(yhat1~experience,data=cps,lty=2)
lines(yhat2~experience,data=cps)
legend("topleft",c("quadratic","spline"),lty=c(2,1),bty="n")

In [ ]:
#BIC
# basic model
cps_plm <- lm(log(wage)~bs(experience,df=5)+education+ethnicity,data=CPS1988)

# 2. we are choosing best model using spline with AIC score
print('----Zadanie 2----')
cps_bs<-lapply(3:10,function(i) lm(log(wage)~bs(experience,df=i)
+education+ethnicity,data=CPS1988))
structure(sapply(cps_bs,BIC),.Names=3:10)
# no to nam mówi który można wybrać, trzeba dać argmin
# 3. 
print('----Zadanie 3')
cps<-data.frame(experience=-2:60, education=with(CPS1988,
mean(education[ethnicity=="cauc"])),ethnicity="cauc")
cps$yhat1<-predict(cps_lm,newdata=cps)
cps$yhat2<-predict(cps_plm,newdata=cps)
plot(log(wage)~jitter(experience,factor=3),pch=19,
col=rgb(0.5,0.5,0.5,alpha=0.02),data=CPS1988)
lines(yhat1~experience,data=cps,lty=2)
lines(yhat2~experience,data=cps)
legend("topleft",c("quadratic","spline"),lty=c(2,1),bty="n")

Zadanie 5

In [ ]:
# basic model
cps_lm<-lm(log(wage)~experience+I(experience^2)+education+ethnicity,data=CPS1988)

# 2. 
cps_int <- lm(log(wage)~experience + I(experience^2)+
education*ethnicity, data=CPS1988)
print(coeftest(cps_int))
# te statystyki nam mówią (to se te testy ilorazowe??) że te ostatnie dwie zmienne nie są aż tak istotne,, w porównaniu do pozostałych, ale wynika to z tego, że są powiązane, po usunięciu jednej druga stanie sie lepiej interpretowalna 

# 3. separacje osobno na każdy poziom
cps_sep <- lm(log(wage)~ethnicity/(experience + I(experience^2)
+ education)-1, data = CPS1988)
print(?summary.lm)

# manualnie 
cps_sep_cf <- matrix(coef(cps_sep),nrow=2)
rownames(cps_sep_cf)<-levels(CPS1988$ethnicity)
colnames(cps_sep_cf)<-names(coef(cps_lm))[1:4]
print(cps_sep_cf)

# 4. jak usatawić manualnie te relevel'e 
CPS1988$region <- relevel(CPS1988$region, ref="south")
cps_region <- lm(log(wage)~ethnicity+education+experience+I(experience^2)
+region,data=CPS1988)